# 01 — Data Collection
## FIFA World Cup 2026 Analytics
---
**Purpose:** Document every data source, API call, and pipeline step so the analysis can be fully reproduced.

**Three data streams:**
1. **Match schedule + results** — ESPN / FIFA.com (all 104 matches); CBS Sports / Wikipedia (MD1–MD2 results)
2. **Historical weather** — Open-Meteo Archive API at each stadium's exact coordinates and UTC kickoff hour
3. **Forecast weather** — Open-Meteo Forecast API for upcoming matches using identical parameters

**Run the pipeline:** `python src/pipeline.py`
**Rebuild the schedule:** `python src/build_dataset.py`


## 0. Setup

In [1]:
%matplotlib inline
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

from pathlib import Path as _Path
_NB_DIR   = _Path.cwd()
BASE      = str(_NB_DIR.parent)
RAW       = str(_NB_DIR.parent / "data" / "raw")
PROCESSED = str(_NB_DIR.parent / "data" / "processed")
FINAL     = str(_NB_DIR.parent / "data" / "final")
EXTERNAL  = str(_NB_DIR.parent / "data" / "external")
print("Ready.")

Ready.


---
## 1. Match Data Sources

### 1a. Completed Matches (MD1 + MD2 — 36 matches)
- **Source:** CBS Sports group-stage pages + Wikipedia
- **Method:** Manual entry into `MATCHES_RAW` tuple in `src/pipeline.py`
- **Output:** `data/raw/match_metadata.csv`, `data/raw/team_match_stats.csv`

### 1b. Full 104-Match Schedule
- **Source:** ESPN fixture article + FIFA.com bracket data
- **Method:** `src/build_dataset.py` constructs from hardcoded `MATCHES` and `STADIUMS` dicts
- **Output:** `data/external/world_cup_matches.csv`, `data/external/match_locations.csv`

### 1c. Stadium Coordinates
- **Source:** Wikipedia decimal-degree listings
- **Coverage:** All 16 host stadiums
- **Note:** Mexico removed DST permanently in 2023; all Mexican venues = UTC-6 year-round


In [2]:
t1  = pd.read_csv(os.path.join(RAW,      "match_metadata.csv"))
t2  = pd.read_csv(os.path.join(RAW,      "team_match_stats.csv"))
ext = pd.read_csv(os.path.join(EXTERNAL, "world_cup_matches.csv"))
loc = pd.read_csv(os.path.join(EXTERNAL, "match_locations.csv"))

print(f"match_metadata.csv    : {t1.shape[0]} rows  (completed group-stage matches)")
print(f"team_match_stats.csv  : {t2.shape[0]} rows  (2 per completed match)")
print(f"world_cup_matches.csv : {ext.shape[0]} rows  (all 104 matches)")
print(f"match_locations.csv   : {loc.shape[0]} stadiums")
print()
print("Match status breakdown:")
print(ext["match_status"].value_counts().to_string())
print()
print("Countries hosting:")
print(ext["country"].value_counts().to_string())
print()
t1[["match_id","match_date","home_team","away_team","stadium","city","venue_type"]].head(4)

match_metadata.csv    : 36 rows  (completed group-stage matches)
team_match_stats.csv  : 72 rows  (2 per completed match)
world_cup_matches.csv : 104 rows  (all 104 matches)
match_locations.csv   : 16 stadiums

Match status breakdown:
match_status
Completed    36
Scheduled    36
TBD          32

Countries hosting:
country
USA       78
Mexico    13
Canada    13



,match_id,match_date,home_team,away_team,stadium,city,venue_type
0,WC2026-001,2026-06-11,Mexico,South Africa,Estadio Azteca,Mexico City,Open
1,WC2026-002,2026-06-11,South Korea,Czechia,Estadio Akron,Guadalajara,Open
2,WC2026-003,2026-06-12,Canada,Bosnia-Herzegovina,BMO Field,Toronto,Open
3,WC2026-004,2026-06-12,USA,Paraguay,SoFi Stadium,Inglewood,Partial


---
## 2. Weather Data — Open-Meteo API

No API key required. Both endpoints are free.

| Endpoint | Use | Data |
|----------|-----|------|
| `archive-api.open-meteo.com/v1/archive` | Completed matches | ERA-5 reanalysis (historical) |
| `api.open-meteo.com/v1/forecast` | Upcoming matches | GFS/ECMWF blend forecast |

**Parameters fetched per match (hourly, extracted at UTC kickoff hour):**
`temperature_2m`, `precipitation`, `relative_humidity_2m`, `dew_point_2m`,
`wind_speed_10m`, `wind_gusts_10m`, `cloud_cover`, `surface_pressure`, `apparent_temperature`

**Derived heat stress metrics calculated in pipeline:**

| Metric | Formula |
|--------|---------|
| `heat_index_c` | NOAA Rothfusz equation (valid T ≥ 27°C, RH ≥ 40%) |
| `wet_bulb_temperature_c` | Stull (2011) approximation |
| `wbgt_c` | `0.7 × WB + 0.3 × T` (shade WBGT) |
| `cooling_break_flag` | 1 if WBGT ≥ 28°C (FIFA protocol) |


In [3]:
# Show API call pattern for one match
print("Archive API call example (Mexico vs South Africa, June 11):")
print()
print("  GET https://archive-api.open-meteo.com/v1/archive")
print("      ?latitude=19.3029")
print("      &longitude=-99.1500")
print("      &start_date=2026-06-11")
print("      &end_date=2026-06-12")
print("      &hourly=temperature_2m,precipitation,relative_humidity_2m,...")
print("      &timezone=UTC")
print()
print("  Kickoff local: 20:00 CST (UTC-6) = 02:00 UTC on June 12")
print("  -> Extract hourly[2] for each parameter")
print()
t3 = pd.read_csv(os.path.join(PROCESSED, "weather_data.csv"))
print(f"Weather records collected : {len(t3)}/{len(t1)} completed matches")
print(f"Missing weather reason    : 8 venues unconfirmed at pipeline run time")
print()
print("Weather value ranges:")
t3[["temperature_c","humidity_percent","precipitation_mm","wbgt_c","heat_index_c"]].describe().round(2)

Archive API call example (Mexico vs South Africa, June 11):

  GET https://archive-api.open-meteo.com/v1/archive
      ?latitude=19.3029
      &longitude=-99.1500
      &start_date=2026-06-11
      &end_date=2026-06-12
      &hourly=temperature_2m,precipitation,relative_humidity_2m,...
      &timezone=UTC

  Kickoff local: 20:00 CST (UTC-6) = 02:00 UTC on June 12
  -> Extract hourly[2] for each parameter

Weather records collected : 36/36 completed matches
Missing weather reason    : 8 venues unconfirmed at pipeline run time

Weather value ranges:


,temperature_c,humidity_percent,precipitation_mm,wbgt_c,heat_index_c
count,28.00,28.00,28.00,28.00,28.00
mean,25.41,58.11,0.19,21.17,26.71
std,4.09,19.39,0.58,3.52,6.07
min,16.80,33.00,0.00,15.97,16.80
25%,22.85,41.75,0.00,18.22,22.85
50%,25.65,53.50,0.00,20.78,25.65
75%,27.98,72.25,0.20,23.46,29.22
max,32.80,97.00,3.00,27.95,40.60


---
## 3. Pipeline Architecture

```
src/pipeline.py
    |
    +-- Step 1  Load match results -> match_metadata.csv, team_match_stats.csv
    |
    +-- Step 2  Fetch archive weather for each completed match
    |           Open-Meteo Archive API at stadium coords + UTC kickoff hour
    |
    +-- Step 3  Calculate heat stress metrics
    |           heat_index_c (Rothfusz), wet_bulb_temperature_c (Stull),
    |           wbgt_c = 0.7*WB + 0.3*T
    |           -> weather_data.csv
    |
    +-- Step 4  Build modeling dataset
    |           Merge match + weather + ELO ratings + rolling goal averages
    |           + Haversine travel distances + venue encoding
    |           -> modeling_dataset.csv
    |
    +-- Step 5  Train and evaluate 3 models x 5 feature sets
    |           5-fold cross-validation -> prediction_results.csv
    |
    +-- Step 6  Forecast all upcoming matches
                Open-Meteo Forecast API + best model
                -> remaining_match_forecasts.csv
```


---
## 4. Limitations and Known Issues

| Issue | Impact |
|-------|--------|
| 8 matches have NULL weather (venue unconfirmed at pipeline run) | n = 28, not 36, for weather analyses |
| ELO ratings are pre-tournament estimates | Team strength not updated after each match |
| Forecast accuracy degrades beyond 7 days | Re-run `pipeline.py` closer to each match |
| Knockout teams are TBD until group stage ends | Predictions for R32+ require team confirmation |
| Covered stadiums (NRG, AT&T, Mercedes-Benz, BC Place) | Outdoor API weather ≠ in-stadium conditions |
| Cards, passes, shots, xG, distance = **synthetic** | Simulated from historical WC distributions; labelled clearly |
